In [186]:
import copy

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error 
from sklearn.neural_network import MLPRegressor 
from sklearn.preprocessing import StandardScaler, OneHotEncoder


from typing import Literal, cast, Any

In [187]:
X = pd.read_csv('train_data.csv').set_index('#')
y = pd.read_csv('ytrain_data.csv').set_index('#')

In [188]:
nominal_columns = [
    'state',
]

In [189]:
X = X[X.index != 10089]
y = y[y.index != 10089]

In [190]:
X[['state']].value_counts()

state
2.0      13581
3.0       4071
1.0       3380
4.0        296
Name: count, dtype: int64

In [191]:
encoder = OneHotEncoder(sparse_output=False, dtype=int, )
encoded_data = encoder.fit_transform(X[nominal_columns])

In [192]:
for i in range(0, encoded_data.shape[1]):
    column_name = f'state_category_{i + 1}'
    X[column_name] = encoded_data[:, i]

X = X.drop(columns=['state'])

In [193]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42
)

In [194]:
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test) 

In [195]:
neurons_options    = [5, 10, 20, 50]
activation_options = ["logistic", "tanh", "relu",]

search_results     = []

In [196]:
for neurons in neurons_options:
    for activation in activation_options: 

        mlp = MLPRegressor(
            hidden_layer_sizes=(neurons,),
            activation=cast(Any, activation),
            solver="sgd",
            max_iter=1000,
            random_state=42
        )

        try:
            mlp.fit(X_train_scaled, y_train.values.ravel())
            y_pred = mlp.predict(X_test_scaled)
            mse = mean_squared_error(y_test, y_pred)

            search_results.append(
                {
                    "Нейроны": neurons,
                    "Активация": activation,
                    "Тестовый MSE": mse
                }
            )

        except Exception as e:
            continue

df_results = pd.DataFrame(search_results).sort_values(by='Тестовый MSE')
print("\nСводная таблица всех проверенных структур: ")
print(df_results.to_string(index=False))


Сводная таблица всех проверенных структур: 
 Нейроны Активация  Тестовый MSE
      10  logistic      0.209890
      50  logistic      0.212344
      20  logistic      0.212517
       5  logistic      0.214078
      20      tanh      0.214946
      50      tanh      0.215661
      10      tanh      0.218663
       5      tanh      0.225680
       5      relu      0.273529
      10      relu      0.304876
      50      relu      0.307672
      20      relu      0.355232


In [204]:
best_config = df_results.iloc[0]

In [205]:
best_neurons = int(best_config["Нейроны"])
best_activation = best_config["Активация"]

In [207]:
global_search_models = []
N_MULTISTART = 10
rng = np.random.default_rng(42)
N_JOG = 3
NOISE_LEVEL = 0.05

In [208]:
for i in range(N_MULTISTART):
    mlp = MLPRegressor(
        hidden_layer_sizes=(best_neurons,),
        activation=best_activation,
        solver="sgd",
        max_iter=2000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=100 + i,
    )
    
    mlp.fit(X_train_scaled, y_train.values.ravel())

    mse = mean_squared_error(y_test.values.ravel(), mlp.predict(X_test_scaled))
    global_search_models.append({"model": mlp, "method": f"Мультистарт #{i+1}",
                                 "mse": mse, "n_iter": mlp.n_iter_})
    print(f"#{i+1} | val MSE: {mse:.5f} | итераций: {mlp.n_iter_}")

#1 | val MSE: 0.21025 | итераций: 165
#2 | val MSE: 0.21279 | итераций: 94
#3 | val MSE: 0.21093 | итераций: 160
#4 | val MSE: 0.21103 | итераций: 108
#5 | val MSE: 0.21040 | итераций: 221
#6 | val MSE: 0.21218 | итераций: 134
#7 | val MSE: 0.20960 | итераций: 210
#8 | val MSE: 0.21079 | итераций: 205
#9 | val MSE: 0.20846 | итераций: 493
#10 | val MSE: 0.21101 | итераций: 210


In [209]:
best = min(global_search_models, key=lambda x: x["mse"])
best_multi_model = best["model"]

In [212]:
for i in range(N_JOG):
    mlp_jog = copy.deepcopy(best_multi_model)
    mlp_jog.set_params(
        warm_start=True,   
        max_iter=500,    
        random_state=200 + i,
    )

    mlp_jog.coefs_ = [c + rng.normal(0, NOISE_LEVEL, c.shape) for c in mlp_jog.coefs_]
    mlp_jog.intercepts_ = [b + rng.normal(0, NOISE_LEVEL, b.shape) for b in mlp_jog.intercepts_]

    mlp_jog.fit(X_train_scaled, y_train.values.ravel())
    
    mse = mean_squared_error(y_test.values.ravel(), mlp_jog.predict(X_test_scaled))
    global_search_models.append(
        {"model": mlp_jog, "method": f"Встряска весов #{i+1}", "mse": mse, "n_iter": mlp_jog.n_iter_}
    )
    print(f" > Встряска #{i+1} | val MSE: {mse:.5f}")

 > Встряска #1 | val MSE: 0.20846
 > Встряска #2 | val MSE: 0.20846
 > Встряска #3 | val MSE: 0.20846


In [211]:
pd.DataFrame(global_search_models).sort_values('mse')

,model,method,mse,n_iter
8,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #9,0.208460,493
10,"MLPRegressor(activation='logistic', early_stop...",Встряска весов #1,0.208460,1
11,"MLPRegressor(activation='logistic', early_stop...",Встряска весов #2,0.208460,1
12,"MLPRegressor(activation='logistic', early_stop...",Встряска весов #3,0.208460,1
6,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #7,0.209602,210
0,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #1,0.210251,165
4,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #5,0.210396,221
7,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #8,0.210795,205
2,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #3,0.210934,160
9,"MLPRegressor(activation='logistic', early_stop...",Мультистарт #10,0.211006,210
